# 03 · Exploratory Data Analysis (EDA)

**Objective:** Discover patterns, distributions, and relationships in the cleaned Amazon Electronics dataset.

Key questions we explore:
1. How are prices and ratings distributed?
2. Which product categories dominate?
3. What correlations exist between numeric features?
4. Does sponsorship affect pricing?

In [ ]:
# ── Imports ──────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
%matplotlib inline

df = pd.read_csv('../data/processed.csv')
print(f'Loaded processed data  →  {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

In [ ]:
# ── Summary statistics ──────────────────────────────
display(df.describe(include='all').T)

In [ ]:
# ── Missing-value heatmap ───────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='viridis', ax=ax)
ax.set_title('Missing-Value Map (yellow = missing)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── Price distribution ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['discounted_price'].dropna(), bins=60, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Discounted Price Distribution')
axes[0].set_xlabel('Price ($)')

# log-scale view for heavily skewed data
sns.histplot(df['discounted_price'].dropna(), bins=60, kde=True, ax=axes[1], color='darkorange', log_scale=True)
axes[1].set_title('Discounted Price Distribution (log scale)')
axes[1].set_xlabel('Price ($) — log')

plt.tight_layout()
plt.show()

In [ ]:
# ── Rating distribution ─────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(df['product_rating'].dropna(), bins=20, kde=True, color='mediumseagreen', ax=ax)
ax.axvline(df['product_rating'].mean(), color='red', ls='--', label=f"Mean = {df['product_rating'].mean():.2f}")
ax.set_title('Product Rating Distribution')
ax.set_xlabel('Rating (out of 5)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Category breakdown ──────────────────────────────
cat_counts = df['product_category'].value_counts()

fig, ax = plt.subplots(figsize=(12, 6))
cat_counts.plot.barh(ax=ax, color=sns.color_palette('Set2', len(cat_counts)))
ax.set_title('Number of Products by Category')
ax.set_xlabel('Count')
ax.set_ylabel('')
ax.invert_yaxis()

for i, v in enumerate(cat_counts):
    ax.text(v + 50, i, f'{v:,}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation heatmap ─────────────────────────────
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

fig, ax = plt.subplots(figsize=(10, 8))
corr = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix (lower triangle)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── Sponsored vs Organic: price comparison ──────────
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x='is_sponsored', y='discounted_price', ax=ax,
            palette={True: 'salmon', False: 'skyblue'})
ax.set_title('Discounted Price: Sponsored vs Organic Listings')
ax.set_xlabel('Is Sponsored')
ax.set_ylabel('Discounted Price ($)')
ax.set_ylim(0, df['discounted_price'].quantile(0.95))  # clip outliers for readability
plt.tight_layout()
plt.show()